# Odysseus GPU LLM + Cloudflare Tunnel

Runs an authenticated OpenAI-compatible Qwen endpoint on the Kaggle GPU and opens a temporary Cloudflare Quick Tunnel for Odysseus. Keep this notebook private and running while you use the model. Kaggle restarts create a new tunnel URL and API key.

Before running: choose a GPU under **Settings → Accelerator** and enable **Internet**.

In [ ]:
import os, re, secrets, subprocess, sys, time
from pathlib import Path
import requests

if not __import__('torch').cuda.is_available():
    raise RuntimeError('Enable a GPU in Kaggle Settings → Accelerator, then restart this session.')

MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
os.environ['ODYSSEUS_MODEL_ID'] = MODEL_ID
os.environ['ODYSSEUS_API_KEY'] = secrets.token_urlsafe(36)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'fastapi>=0.115', 'uvicorn>=0.30', 'transformers>=4.45,<5', 'accelerate>=1.0'], check=True)

script_url = 'https://raw.githubusercontent.com/Mhdkenzz/odysseus/dev/scripts/kaggle_llm_server.py'
script_path = Path('/kaggle/working/kaggle_llm_server.py')
response = requests.get(script_url, timeout=30)
response.raise_for_status()
script_path.write_bytes(response.content)

cloudflared = Path('/kaggle/working/cloudflared')
subprocess.run(['wget', '-q', '-O', str(cloudflared), 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'], check=True)
cloudflared.chmod(0o755)

server_log = open('/kaggle/working/odysseus-llm.log', 'w')
server = subprocess.Popen([sys.executable, str(script_path)], stdout=server_log, stderr=subprocess.STDOUT, start_new_session=True)
tunnel_log = open('/kaggle/working/cloudflared.log', 'w')
tunnel = subprocess.Popen([str(cloudflared), 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8000'], stdout=tunnel_log, stderr=subprocess.STDOUT, start_new_session=True)

print('GPU:', __import__('torch').cuda.get_device_name(0))
print('Starting model and tunnel. The model download may take several minutes.')
deadline = time.time() + 180
base_url = None
while time.time() < deadline:
    tunnel_text = Path('/kaggle/working/cloudflared.log').read_text(errors='ignore')
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', tunnel_text)
    if match:
        base_url = match.group(0) + '/v1'
        break
    if tunnel.poll() is not None:
        raise RuntimeError('Cloudflare Tunnel exited. See /kaggle/working/cloudflared.log')
    time.sleep(2)
if not base_url:
    raise TimeoutError('Tunnel URL did not appear. See /kaggle/working/cloudflared.log')

print('Odysseus Base URL:', base_url, flush=True)
print('Waiting for the model to load...', flush=True)
ready_deadline = time.time() + 1200
while time.time() < ready_deadline:
    if server.poll() is not None:
        print(Path('/kaggle/working/odysseus-llm.log').read_text(errors='ignore')[-5000:])
        raise RuntimeError('Model server exited; inspect its log above.')
    try:
        health = requests.get('http://127.0.0.1:8000/health', timeout=3)
        if health.ok:
            break
    except requests.RequestException:
        pass
    time.sleep(10)
else:
    raise TimeoutError('Model did not become ready; inspect /kaggle/working/odysseus-llm.log')

print('READY — add this endpoint in Odysseus Settings → Models.')
print('Base URL:', base_url)
print('Model:', MODEL_ID)
print('API key:', os.environ['ODYSSEUS_API_KEY'])